# Exercise 11: foundation model embeddings for real promoters

Predict whether a **251-base human DNA sequence is a non-TATA promoter**. Promoters are regulatory regions associated with transcription initiation; this benchmark focuses on promoters without a TATA box. Label **1** means promoter and label **0** means non-promoter. All data are downloaded here; no earlier notebook or uploaded file is needed.

Adapted from Andrea Gariboldi's [bioFoundationModels notebook](https://github.com/andrea-gariboldi/bioFoundationModels/blob/main/bioFoundation_models.ipynb), especially its tokenization and hidden-state examples. We use the same [50M Nucleotide Transformer v2](https://huggingface.co/InstaDeepAI/nucleotide-transformer-v2-50m-multi-species).

A foundation model is pretrained on a large collection of data and can supply representations for downstream tasks. This transformer learned from DNA in 850 species by predicting masked tokens. An **embedding** is a vector of numerical features. Here each DNA token receives 512 features that depend on its surrounding sequence.

Compare two paths using the same pretrained representation:

- **A: DNA → frozen transformer → mean over DNA tokens → logistic regression.**
- **B: DNA → frozen transformer → one convolutional layer over DNA tokens → ReLU → max pooling → linear classifier.**

The transformer stays fixed in both paths. We train only the downstream classifiers on promoter labels. The comparison asks whether a learned combination of neighboring token vectors helps more than a simple sequence average. Use validation data to make choices, then evaluate both frozen choices on the official test set. **Run all includes official test evaluation.**


## Setup

In Colab, select **Runtime → Change runtime type → T4 GPU** before running. A GPU accelerates transformer inference; a CPU also works but is much slower for all 36,131 sequences. The first run needs internet access to download the dataset, model weights and custom model code.

`%pip` installs into the notebook's Python environment. `-q` reduces installation messages. `pyarrow` reads the dataset's Parquet files; `transformers==4.57.1` pins the model-loading library to the version used in the source notebook. Colab normally supplies PyTorch, NumPy, pandas, scikit-learn and Matplotlib. For local use, install the repository's `requirements.txt` first.

We cache token embeddings in CPU RAM so CNN epochs do not repeat expensive transformer inference. The complete dataset needs roughly **2 GB for cached token features**, plus the model and other arrays. We store those features with 16-bit precision and convert each CNN batch to 32-bit precision for training.


In [ ]:
%pip install -q pyarrow transformers==4.57.1


### Choose reproducible starting settings

The libraries below handle arrays (`numpy`), tables (`pandas`), plots (`matplotlib`), neural networks (`torch`), pretrained models (`transformers`) and logistic regression/metrics (`sklearn`). `deepcopy` will save an independent copy of the best CNN weights.

| Setting | Meaning and reason for this choice |
| --- | --- |
| `seed = 42` | Fixes validation sampling and PyTorch's random initialization, dropout and batch shuffling. The value is arbitrary; keeping it fixed makes comparisons easier. Exact results can still differ across hardware. |
| `batch_size = 32` | Number of sequences sent through the transformer together. A modest batch limits GPU memory use; reduce it if memory runs out. It does not change the classifier's learning rate. |
| `cnn_batch_size = 128` | Number of cached examples per classifier update. The small CNN needs much less memory than the transformer, so it can use larger batches. |
| `max_epochs = 15` | Maximum number of passes through CNN training data. We restore the epoch with lowest validation loss, so the last epoch is not automatically selected. |
| `learning_rate = 0.001` | Initial Adam step size for the CNN. This is a conventional starting point, not a claim that it is optimal. |
| `threshold = 0.5` | Predict promoter when its probability is at least 0.5. Keep this fixed for both models; any threshold tuning must use validation data. |
| `model_name`, `model_revision` | Choose the relatively small 50M model and an exact Hugging Face checkpoint. The revision identifies both weights and custom code so future repository updates do not silently change our model. |

`torch.manual_seed` applies the seed to PyTorch. `torch.set_num_threads(2)` limits CPU parallelism for this teaching example; it is a resource setting, not a model parameter. `cuda` selects an NVIDIA GPU when available, otherwise `cpu`. `.to(device)` later moves model weights or batches to that processor.


In [ ]:
from copy import deepcopy

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from transformers import AutoTokenizer, AutoModelForMaskedLM
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, average_precision_score, log_loss,
    RocCurveDisplay, PrecisionRecallDisplay, ConfusionMatrixDisplay,
)

seed = 42
batch_size = 32
cnn_batch_size = 128
max_epochs = 15
learning_rate = 0.001
threshold = 0.5
model_name = 'InstaDeepAI/nucleotide-transformer-v2-50m-multi-species'
model_revision = '81b29e5786726d891dbf929404ef20adca5b36f1'

torch.manual_seed(seed)
torch.set_num_threads(2)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)


## 1. Download promoters and reserve validation data

Use the [Genomic Benchmarks human non-TATA promoter dataset](https://huggingface.co/datasets/katarinagresova/Genomic_Benchmarks_human_nontata_promoters). The URL contains an exact dataset revision. Parquet is a compressed table format; `read_parquet` downloads and reads it. Rename `seq` to `sequence` to make its meaning clear.

The assertions check that every sequence has exactly 251 characters from A/C/G/T/N and that both labels are present. `N` means an unknown base; keep it rather than dropping an example. The pattern `[ACGTN]{251}` specifies the allowed bases and length, and `fullmatch` checks the complete string.

`groupby('label').sample(frac=0.20, random_state=seed)` takes 20% of each class for validation, preserving class proportions. `drop(validation_data.index)` makes the remaining rows the training set. We retain original row indices so the split can be checked. `to_numpy()` turns the labels into arrays for the classifiers.

This gives **21,678 training / 5,419 validation / 9,034 official test sequences**. Training data fit parameters; validation data guide choices and CNN epoch selection; the official test set estimates final performance. Download test data only in the last section. This existing benchmark split is not a chromosome-held-out experiment.


In [ ]:
data_url = (
    'https://huggingface.co/datasets/katarinagresova/'
    'Genomic_Benchmarks_human_nontata_promoters/resolve/'
    'e0003669df0a180c8570ebe091ed91f20fa080ec/data/'
)
benchmark_train = pd.read_parquet(data_url + 'train-00000-of-00001-aeb55728756a44ae.parquet')
benchmark_train = benchmark_train.rename(columns={'seq': 'sequence'})
assert benchmark_train['sequence'].str.fullmatch('[ACGTN]{251}').all()
assert set(benchmark_train['label']) == {0, 1}

validation_data = benchmark_train.groupby('label').sample(frac=0.20, random_state=seed)
train_data = benchmark_train.drop(validation_data.index)
y_train = train_data['label'].to_numpy()
y_validation = validation_data['label'].to_numpy()
print('Training:', len(train_data))
print('Validation:', len(validation_data))
print('Training class counts:')
print(train_data['label'].value_counts())


## 2. Load and freeze the foundation model

`AutoTokenizer` loads the checkpoint's DNA-to-token mapping. `AutoModelForMaskedLM` loads the architecture originally used for masked-token prediction. We will use its final hidden states as features; its vocabulary predictions are not promoter probabilities. The final layer supplies a **512-dimensional contextual embedding per token**. A vector coordinate is a learned feature, not a named nucleotide or a guaranteed biological property.

`from_pretrained` retrieves the existing weights instead of initializing a new transformer. `trust_remote_code=True` permits the checkpoint's custom Python architecture to load; use it for a checkpoint whose code you trust. `revision=model_revision` pins that code and its weights.

Three settings have separate jobs: **`requires_grad_(False)`** freezes parameters, **`eval()`** selects inference behavior for layers such as dropout, and **`torch.inference_mode()`** later avoids building gradient-tracking information. `eval()` alone does not freeze weights. We print the number of trainable transformer parameters; it should be zero. Neither classifier updates this model.


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(
    model_name, revision=model_revision, trust_remote_code=True,
)
foundation_model = AutoModelForMaskedLM.from_pretrained(
    model_name, revision=model_revision, trust_remote_code=True,
).to(device)
foundation_model.requires_grad_(False)
foundation_model.eval()
print('Embedding dimensions:', foundation_model.config.hidden_size)
print('Trainable transformer parameters:',
      sum(p.numel() for p in foundation_model.parameters() if p.requires_grad))


## 3. Inspect DNA tokens

A **token** is one item in the model's input vocabulary. This tokenizer uses non-overlapping 6-mers where possible, with single bases for remaining positions and `N`. It adds a `<cls>` marker before the DNA. Therefore 251 bases do not give 251 tokens: ordinary A/C/G/T examples here have 46 DNA tokens plus `<cls>`. An `N` can change how a sequence is divided into tokens.

`iloc[0]` selects one training example without using its label. `return_tensors='pt'` requests PyTorch tensors. `input_ids` are vocabulary indices, not embeddings yet. `convert_ids_to_tokens` converts them back to readable strings so we can inspect the exact inputs. Shape `(1, tokens)` means one example and its token positions.

When sequences are combined into a batch, shorter token lists need padding. An **attention mask** tells the transformer which positions are real inputs. Our separate **DNA mask** will also exclude `<cls>` and other special tokens when pooling or applying the CNN. Keep `N` as a valid DNA token; it is not padding.


In [ ]:
example_sequence = train_data.iloc[0]['sequence']
example_inputs = tokenizer(example_sequence, return_tensors='pt')
example_tokens = tokenizer.convert_ids_to_tokens(example_inputs['input_ids'][0].tolist())
print('DNA:', example_sequence)
print('Model tokens:', example_tokens)
print('Input shape (batch, tokens):', tuple(example_inputs['input_ids'].shape))


## 4. Extract shared features for both classifiers

The function returns three objects in the same row order as its input: **mean embeddings** `(examples, 512)` for logistic regression, **token embeddings** `(examples, token positions, 512)` for the CNN, and a **DNA mask** `(examples, token positions)` that marks usable positions. No labels enter this function and nothing is fitted here.

- `padding=True` pads to the longest tokenized sequence in this split. Tokenizing the split once gives every stored row the same width. `truncation=False` keeps every base. These short sequences fit the checkpoint's context limit. Only one inference batch moves to the GPU at a time.
- `torch.isin(input_ids, special_ids)` identifies markers from their actual IDs. Combining its inverse with the attention mask keeps DNA positions, including `N`, and removes special/padding positions.
- `output_hidden_states=True` exposes the intermediate representations; `hidden_states[-1]` selects the last transformer layer. It does not select vocabulary logits or one vector for the whole sequence.
- For path A, sum DNA vectors and divide by the number of DNA tokens. `unsqueeze(-1)` adds an axis so one mask value applies to all 512 features. Each token gets equal weight, even if it represents one base rather than six. Averaging compresses positional information, although each vector already contains sequence context.
- For path B, keep ordered vectors and zero the excluded positions. Store this cache as `float16` to halve RAM use; this introduces small rounding differences. CNN batches are converted to `float32` before training. The model itself uses its default 32-bit precision.

`torch.empty` reserves CPU storage that we fill batch by batch. `np.concatenate` joins the small arrays of mean vectors. The progress message reports completed sequences; `end='\r'` updates one line. Extract training and validation features now; test features wait until the last section.


In [ ]:
def embed_sequences(sequences):
    sequences = list(sequences)
    inputs = tokenizer(sequences, padding=True, truncation=False, return_tensors='pt')
    special_ids = torch.tensor(tokenizer.all_special_ids)
    special_tokens = torch.isin(inputs['input_ids'], special_ids)
    dna_mask = inputs['attention_mask'].bool() & ~special_tokens
    token_features = torch.empty(
        len(sequences), inputs['input_ids'].shape[1],
        foundation_model.config.hidden_size, dtype=torch.float16,
    )
    batches = []
    with torch.inference_mode():
        for start in range(0, len(sequences), batch_size):
            stop = min(start + batch_size, len(sequences))
            batch_inputs = {name: values[start:stop].to(device) for name, values in inputs.items()}
            outputs = foundation_model(**batch_inputs, output_hidden_states=True)
            token_embeddings = outputs.hidden_states[-1]
            mask = dna_mask[start:stop].to(device).unsqueeze(-1)
            weights = mask.to(token_embeddings.dtype)
            pooled = (token_embeddings * weights).sum(dim=1) / weights.sum(dim=1)
            batches.append(pooled.cpu().numpy())
            token_features[start:stop] = token_embeddings.masked_fill(~mask, 0).cpu().half()
            print(f'Embedded {min(start + batch_size, len(sequences))}/{len(sequences)}',
                  end='\r', flush=True)
    print()
    return np.concatenate(batches).astype(np.float32), token_features, dna_mask

x_train, train_tokens, train_mask = embed_sequences(train_data['sequence'])
x_validation, validation_tokens, validation_mask = embed_sequences(validation_data['sequence'])
print('Training mean embeddings:', x_train.shape)
print('Training token embeddings:', tuple(train_tokens.shape))
print('Validation token embeddings:', tuple(validation_tokens.shape))


## 5. Path A: mean embeddings + logistic regression

Logistic regression computes a weighted sum of the 512 features, adds an intercept and applies sigmoid to produce a promoter probability. This provides a small, easy-to-explain baseline for the pretrained representation.

`StandardScaler()` learns one training mean and standard deviation per feature, then subtracts the mean and divides by the standard deviation. Features with different numerical ranges can otherwise receive unequal regularization. `make_pipeline` makes fitting and later prediction apply these steps in order. Validation and test data reuse the training scaler; fitting their own scalers would leak information.

`LogisticRegression(C=1.0, max_iter=2000, random_state=seed)` uses the default L2 penalty and `lbfgs` solver. **`C` is inverse regularization strength**: smaller values penalize large weights more strongly. We choose 1.0 as a fixed baseline. `max_iter=2000` allows the numerical optimizer enough iterations to converge; it is a limit on optimization iterations, not neural-network epochs. `random_state` records our seed convention, although the default solver does not depend on it.

Only `.fit(x_train, y_train)` learns parameters. `n_features_in_` confirms that each example supplies 512 features. Keep this classifier fixed while training path B.


In [ ]:
classifier = make_pipeline(
    StandardScaler(),
    LogisticRegression(C=1.0, max_iter=2000, random_state=seed),
)
classifier.fit(x_train, y_train)
print('Classifier input dimensions:', classifier.n_features_in_)


## 6. Path B: one CNN layer over token embeddings

Keep the token axis so the CNN can scan neighboring vectors. Convolving over the 512 coordinates of an already-averaged vector would have no meaningful DNA neighborhood. Here each convolution sees **three adjacent token positions and all 512 features at each position**. Usually three 6-mer tokens span 18 bases, but single-base tokens make this span variable. The embeddings already contain broader context, so this is not a strictly 18-base biological detector.

| Layer / choice | Meaning and reason |
| --- | --- |
| `Conv1d(512, 32, kernel_size=3, padding=0)` | Learn 32 different local combinations. Three tokens give a small neighborhood; 32 filters keep the classifier small (about 49,000 trainable parameters). `padding=0` creates only complete windows, reducing the length from L to L−2. PyTorch enables a learned bias for each filter by default. |
| ReLU | Replace negative filter responses with zero, adding a nonlinear decision step. |
| Mask + global max pooling | Keep the strongest valid response per filter, giving 32 features regardless of token count. This asks whether a useful pattern occurs somewhere and discards its absolute position. |
| `Dropout(0.3)` | During training, randomly zero 30% of pooled features to discourage reliance on just a few responses. It is disabled during evaluation. |
| `Linear(32, 1)` | Combine the pooled features into one logit. A logit is an unrestricted score; sigmoid converts it into a probability. The linear layer also learns an intercept. |

`transpose(1, 2)` changes `(batch, tokens, features)` to the `(batch, channels, length)` format required by `Conv1d`. The features are channels, and the convolution moves along tokens. We use the pretrained feature scale directly for the CNN; the logistic-regression scaler is specific to path A.

Zeroing padding embeddings is not enough: a convolutional bias or a partial window could still produce a strong response. `mask.unfold(1, 3, 1).all(dim=-1)` marks only windows whose **three positions are all DNA**. Fill other responses with negative infinity before max pooling so they cannot win. This also excludes windows containing `<cls>` and preserves genuine `N` tokens. The short class below keeps this masking next to the layers it controls.

`BCEWithLogitsLoss` calculates binary cross-entropy directly from logits, including sigmoid internally for numerical stability. **Do not add sigmoid inside the model.** `Adam` updates only `cnn.parameters()` with learning rate 0.001. `weight_decay=0.0001` adds a small penalty on large parameters; it is a different regularization control from logistic regression's `C`. These are starting settings chosen before test evaluation.


In [ ]:
class TokenCNN(nn.Module):
    def __init__(self, embedding_size):
        super().__init__()
        self.convolution = nn.Conv1d(embedding_size, 32, kernel_size=3, padding=0)
        self.dropout = nn.Dropout(0.3)
        self.output = nn.Linear(32, 1)

    def forward(self, tokens, mask):
        responses = torch.relu(self.convolution(tokens.transpose(1, 2)))
        valid_windows = mask.unfold(1, 3, 1).all(dim=-1)
        responses = responses.masked_fill(~valid_windows.unsqueeze(1), -torch.inf)
        pooled = responses.amax(dim=2)
        return self.output(self.dropout(pooled)).squeeze(1)

cnn = TokenCNN(foundation_model.config.hidden_size).to(device)
loss_function = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(cnn.parameters(), lr=learning_rate, weight_decay=0.0001)
print(cnn)
print('Trainable CNN parameters:', sum(p.numel() for p in cnn.parameters() if p.requires_grad))


### Make batches from cached features

`TensorDataset` keeps a sequence's token matrix, mask and label together. Labels use `float32` because binary cross-entropy expects floating-point targets (0.0 or 1.0). Token matrices remain on CPU in `float16` until each training batch is converted.

`DataLoader(..., batch_size=cnn_batch_size)` divides that dataset into batches. `shuffle=True` changes training order each epoch to avoid always seeing examples in the same order. Validation uses `shuffle=False` so probabilities remain aligned with labels. The defaults use one loading process (`num_workers=0`) and retain the final smaller batch (`drop_last=False`); extra workers are unnecessary for in-memory tensors. No validation or test examples enter the training loader.


In [ ]:
train_loader = DataLoader(
    TensorDataset(train_tokens, train_mask, torch.tensor(y_train, dtype=torch.float32)),
    batch_size=cnn_batch_size, shuffle=True,
)
validation_loader = DataLoader(
    TensorDataset(validation_tokens, validation_mask, torch.tensor(y_validation, dtype=torch.float32)),
    batch_size=cnn_batch_size, shuffle=False,
)


## 7. Train the CNN and select its epoch

An **epoch** is one pass through all training examples. `cnn.train()` enables dropout. For each batch, `zero_grad()` clears previous gradients, the forward pass calculates logits, `loss.backward()` computes gradients, and `optimizer.step()` updates the convolution and output layer. Cached transformer features are fixed inputs, so backpropagation stops at this classifier.

`.to(device, dtype=torch.float32)` transfers the small feature batch to the chosen processor and restores 32-bit arithmetic. Mask values remain Boolean. The loss is averaged within a batch; multiplying by its number of labels before summing gives a correctly weighted epoch average, including a smaller last batch.

After each epoch, `cnn.eval()` disables dropout and `torch.no_grad()` disables gradient tracking. Validation performs no optimizer steps. Save weights whenever validation loss improves; `deepcopy` prevents the saved tensors from changing during later updates. After at most 15 epochs, restore the best checkpoint with `load_state_dict`. We do not stop training early here: all configured epochs run, then the best is selected.

The plot uses epoch on the horizontal axis and binary cross-entropy on the vertical axis (lower is better). A widening train/validation gap can indicate overfitting. Training loss includes dropout and changing weights; validation uses fixed weights without dropout, so the two curves are not measured under identical conditions. The dashed line marks the selected epoch.


In [ ]:
training_losses = []
validation_losses = []
best_validation_loss = np.inf
for epoch in range(1, max_epochs + 1):
    cnn.train()
    total_loss = 0.0
    for tokens, mask, labels in train_loader:
        tokens = tokens.to(device, dtype=torch.float32)
        mask, labels = mask.to(device), labels.to(device)
        optimizer.zero_grad()
        logits = cnn(tokens, mask)
        loss = loss_function(logits, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(labels)
    training_loss = total_loss / len(train_data)

    cnn.eval()
    total_loss = 0.0
    with torch.no_grad():
        for tokens, mask, labels in validation_loader:
            logits = cnn(tokens.to(device, dtype=torch.float32), mask.to(device))
            loss = loss_function(logits, labels.to(device))
            total_loss += loss.item() * len(labels)
    validation_loss = total_loss / len(validation_data)
    training_losses.append(training_loss)
    validation_losses.append(validation_loss)
    if validation_loss < best_validation_loss:
        best_validation_loss = validation_loss
        best_epoch = epoch
        best_state = deepcopy(cnn.state_dict())
    print(f'Epoch {epoch}: training={training_loss:.4f}, validation={validation_loss:.4f}', flush=True)

cnn.load_state_dict(best_state)
cnn.eval()
print('Selected epoch:', best_epoch)
plt.figure(figsize=(8, 4))
plt.plot(range(1, max_epochs + 1), training_losses, label='Training (with dropout)')
plt.plot(range(1, max_epochs + 1), validation_losses, label='Validation')
plt.axvline(best_epoch, color='black', linestyle='--', label='Selected epoch')
plt.xlabel('Epoch')
plt.ylabel('Binary cross-entropy')
plt.legend()
plt.tight_layout()
plt.show()


## 8. Compare validation performance

Both paths use the same examples and pretrained transformer. Their main differences are how they combine token features and how many downstream parameters they learn. A CNN advantage would support this combination of choices; it would not isolate convolution from parameter count, pooling or optimization. Neither path is guaranteed to win.

`predict_proba(... )[:, 1]` selects logistic regression's probability for class 1 (the known classes are 0 and 1). The CNN prediction function collects sigmoid-transformed logits in loader order. `eval()` and `no_grad()` make this inference without training.

| Metric | Interpretation |
| --- | --- |
| Accuracy | Fraction of all labels predicted correctly at threshold 0.5. Higher is better. |
| Balanced accuracy | Mean of the two class recalls, giving each class equal importance. Higher is better. |
| Precision | Among predicted promoters, the fraction that are promoters. |
| Recall | Among actual promoters, the fraction detected. |
| F1 | Harmonic mean of precision and recall; high only when both are high. |
| ROC AUC | Measures ranking across thresholds using true-positive and false-positive rates. Random ranking gives about 0.5. |
| Average precision | Summarizes the precision–recall relationship across thresholds; useful when the positive class is uncommon. Higher is better. |
| Log loss | Penalizes inaccurate probabilities, especially confident mistakes. Lower is better. |

`zero_division=0` reports zero instead of a warning if precision/recall/F1 has no defined denominator. It does not make that prediction behavior desirable. Accuracy, balanced accuracy, precision, recall and F1 depend on our chosen threshold; AUC and average precision summarize probability rankings. Log loss uses the probabilities directly. `round(4)` affects display only.

The ROC plot includes a random-ranking diagonal. The precision–recall plot includes the split's promoter fraction as a reference for random ranking. The confusion matrix uses label order `[0, 1]` (non-promoter, promoter); rows are actual labels and columns are predictions. Each model gets the same three panels. `figsize`, colors and `tight_layout()` control readability, not predictions.

Discuss what is learned during pretraining versus classifier training, and what mean versus max pooling retains. Both pooling operations discard absolute position. Pretraining may include related genomic sequences, so this benchmark does not establish generalization to unseen genomes. Contextual CNN filters are not automatically identifiable DNA motifs, and predictive performance alone does not establish a promoter mechanism.


In [ ]:
def predict_cnn_probabilities(loader):
    cnn.eval()
    batches = []
    with torch.no_grad():
        for tokens, mask, _ in loader:
            logits = cnn(tokens.to(device, dtype=torch.float32), mask.to(device))
            batches.append(torch.sigmoid(logits).cpu().numpy())
    return np.concatenate(batches)

def calculate_metrics(labels, probabilities):
    predictions = (probabilities >= threshold).astype(int)
    return pd.Series({
        'accuracy': accuracy_score(labels, predictions),
        'balanced_accuracy': balanced_accuracy_score(labels, predictions),
        'precision': precision_score(labels, predictions, zero_division=0),
        'recall': recall_score(labels, predictions, zero_division=0),
        'f1': f1_score(labels, predictions, zero_division=0),
        'roc_auc': roc_auc_score(labels, probabilities),
        'average_precision': average_precision_score(labels, probabilities),
        'log_loss': log_loss(labels, probabilities),
    })

def plot_evaluation(labels, probabilities, title):
    fig, axes = plt.subplots(1, 3, figsize=(16, 4))
    RocCurveDisplay.from_predictions(labels, probabilities, ax=axes[0])
    axes[0].plot([0, 1], [0, 1], 'k--')
    PrecisionRecallDisplay.from_predictions(labels, probabilities, ax=axes[1])
    axes[1].axhline(np.mean(labels), color='black', linestyle='--')
    ConfusionMatrixDisplay.from_predictions(
        labels, probabilities >= threshold, labels=[0, 1],
        display_labels=['Non-promoter', 'Promoter'],
        ax=axes[2], cmap='Blues', colorbar=False,
    )
    fig.suptitle(title)
    plt.tight_layout()
    plt.show()

validation_probabilities = {
    'Mean + logistic regression': classifier.predict_proba(x_validation)[:, 1],
    'Token CNN': predict_cnn_probabilities(validation_loader),
}
validation_results = pd.DataFrame({
    name: calculate_metrics(y_validation, probabilities)
    for name, probabilities in validation_probabilities.items()
}).T
print('Validation metrics:')
print(validation_results.round(4).to_string())
for name, probabilities in validation_probabilities.items():
    plot_evaluation(y_validation, probabilities, 'Validation: ' + name)


## 9. STOP before official test evaluation

Finish all choices using validation data first, including regularization, CNN settings, epoch selection and threshold. Then download the unchanged official test set and extract both feature forms in one transformer pass per batch. Preserve every row, including the sequence containing `N`.

The logistic-regression pipeline reuses its fitted training scaler and coefficients. The CNN reuses its selected validation checkpoint. The test loader has `shuffle=False`, preserving alignment with `y_test`. We evaluate both predeclared paths on the same test examples without updating either model.

The output is a two-row table of the same eight metrics and evaluation plots for each path. Do not use this comparison to tune settings further; that would turn the test set into another validation set. There is no refit on the combined training and validation data in this notebook. **Run all includes this cell.**


In [ ]:
test_data = pd.read_parquet(data_url + 'test-00000-of-00001-d7614d9116c62f44.parquet')
test_data = test_data.rename(columns={'seq': 'sequence'})
assert test_data['sequence'].str.fullmatch('[ACGTN]{251}').all()
assert set(test_data['label']) == {0, 1}
x_test, test_tokens, test_mask = embed_sequences(test_data['sequence'])
y_test = test_data['label'].to_numpy()
test_loader = DataLoader(
    TensorDataset(test_tokens, test_mask, torch.tensor(y_test, dtype=torch.float32)),
    batch_size=cnn_batch_size, shuffle=False,
)
test_probabilities = {
    'Mean + logistic regression': classifier.predict_proba(x_test)[:, 1],
    'Token CNN': predict_cnn_probabilities(test_loader),
}
test_results = pd.DataFrame({
    name: calculate_metrics(y_test, probabilities)
    for name, probabilities in test_probabilities.items()
}).T
print('Official test sequences:', len(test_data))
print(test_results.round(4).to_string())
for name, probabilities in test_probabilities.items():
    plot_evaluation(y_test, probabilities, 'Official test: ' + name)
